For this work, KFUPM & King Abdulaziz Center for World Culture has been selected.

In [1]:
%pip install -q osmnx folium networkx shapely pyproj

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import osmnx as ox
import networkx as nx
import folium
import time
import random
import math

In [3]:
# Coordinates: (lat, lon)
kfupm = (26.3066, 50.1450)
ithra = (26.3278, 50.1545)

# Map center
center = ((kfupm[0] + ithra[0]) / 2,
          (kfupm[1] + ithra[1]) / 2)

print("KFUPM:", kfupm)
print("Ithra:", ithra)
print("Center:", center)
def base_map(center, kfupm, ithra):
    m = folium.Map(location=center, zoom_start=14)
    folium.Marker(kfupm, popup="KFUPM", icon=folium.Icon(color="blue")).add_to(m)
    folium.Marker(ithra, popup="Ithra", icon=folium.Icon(color="green")).add_to(m)
    return m

m0 = base_map(center, kfupm, ithra)
m0

KFUPM: (26.3066, 50.145)
Ithra: (26.3278, 50.1545)
Center: (26.3172, 50.14975)


Graph

In [6]:

ox.settings.use_cache = True
ox.settings.log_console = True

# Current OSMnx settings
ox.settings.overpass_url = "https://overpass.private.coffee/api"
ox.settings.overpass_rate_limit = False
ox.settings.requests_timeout = 300

# Smaller query area
G = ox.graph_from_point(
    center,
    dist=2000,
    network_type="drive",
    simplify=True
)

print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

Nodes: 1154
Edges: 2841


Nearest Nodes & Helper Functions

In [7]:
src = ox.distance.nearest_nodes(G, X=kfupm[1], Y=kfupm[0])
dst = ox.distance.nearest_nodes(G, X=ithra[1], Y=ithra[0])

print("Source node:", src)
print("Destination node:", dst)
def path_length_m(G, path):
    """Total path length in meters, using edge 'length'."""
    total = 0.0
    for u, v in zip(path[:-1], path[1:]):
        data = G.get_edge_data(u, v)
        best = min(attr.get("length", float("inf")) for attr in data.values())
        total += best
    return total

def add_route_to_map(m, G, path, name, color=None):
    """Add route to map."""
    coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in path]
    folium.PolyLine(coords, tooltip=name, weight=5, color=color).add_to(m)
    return m

Source node: 6582857339
Destination node: 1489746234


BFS

In [8]:
start = time.perf_counter()
bfs_path = nx.shortest_path(G, source=src, target=dst)
bfs_time = time.perf_counter() - start
bfs_dist = path_length_m(G, bfs_path)

print(f"BFS time: {bfs_time:.6f} s")
print(f"BFS distance: {bfs_dist:.0f} m")

m_bfs = base_map(center, kfupm, ithra)
m_bfs = add_route_to_map(m_bfs, G, bfs_path, "BFS", color="blue")
m_bfs

BFS time: 0.000458 s
BFS distance: 6492 m


DFS

In [9]:
def dfs_path_find(G, source, target):
    stack = [(source, [source])]
    visited = set()

    while stack:
        node, path = stack.pop()
        if node == target:
            return path
        if node in visited:
            continue
        visited.add(node)

        for nbr in G.successors(node):
            if nbr not in visited:
                stack.append((nbr, path + [nbr]))

    raise nx.NetworkXNoPath("DFS did not find a path")

start = time.perf_counter()
dfs_path = dfs_path_find(G, src, dst)
dfs_time = time.perf_counter() - start
dfs_dist = path_length_m(G, dfs_path)

print(f"DFS time: {dfs_time:.6f} s")
print(f"DFS distance: {dfs_dist:.0f} m")

m_dfs = base_map(center, kfupm, ithra)
m_dfs = add_route_to_map(m_dfs, G, dfs_path, "DFS", color="red")
m_dfs

DFS time: 0.001104 s
DFS distance: 37514 m


Dijkstra

In [10]:
start = time.perf_counter()
dijkstra_path = nx.shortest_path(G, source=src, target=dst, weight="length", method="dijkstra")
dijkstra_time = time.perf_counter() - start
dijkstra_dist = path_length_m(G, dijkstra_path)

print(f"Dijkstra time: {dijkstra_time:.6f} s")
print(f"Dijkstra distance: {dijkstra_dist:.0f} m")

m_dij = base_map(center, kfupm, ithra)
m_dij = add_route_to_map(m_dij, G, dijkstra_path, "Dijkstra", color="green")
m_dij

Dijkstra time: 0.001704 s
Dijkstra distance: 6243 m


Hill Climbing with Backtracking

In [13]:
import time
import networkx as nx
from IPython.display import display


def hill_climbing_backtracking_path(G, source, target):
    """
    Hill climbing with backtracking.
    Neighbors closer to the target are explored first.
    """

    visited = {source}
    parent = {source: None}
    stack = [source]

    while stack:
        current = stack.pop()

        if current == target:
            path = []
            node = target

            while node is not None:
                path.append(node)
                node = parent[node]

            return path[::-1]

        neighbors = [
            node
            for node in G.successors(current)
            if node not in visited
        ]

        # Explore nodes closest to the target first
        neighbors.sort(
            key=lambda node: straight_line_distance(G, node, target),
            reverse=True
        )

        for neighbor in neighbors:
            visited.add(neighbor)
            parent[neighbor] = current
            stack.append(neighbor)

    raise nx.NetworkXNoPath(
        "Hill climbing with backtracking could not find a route."
    )


start = time.perf_counter()

hill_path = hill_climbing_backtracking_path(G, src, dst)

hill_time = time.perf_counter() - start
hill_dist = path_length_m(G, hill_path)

print(f"Hill Climbing time: {hill_time:.6f} s")
print(f"Hill Climbing distance: {hill_dist:.0f} m")
print(f"Hill Climbing nodes: {len(hill_path)}")

m_hill = base_map(center, kfupm, ithra)

m_hill = add_route_to_map(
    m_hill,
    G,
    hill_path,
    "Hill Climbing with Backtracking",
    color="orange"
)

display(m_hill)

Hill Climbing time: 0.000387 s
Hill Climbing distance: 12346 m
Hill Climbing nodes: 108


A*

In [14]:
import time
import networkx as nx
from IPython.display import display


start = time.perf_counter()

astar_path = nx.astar_path(
    G,
    source=src,
    target=dst,
    heuristic=lambda node, target: straight_line_distance(
        G, node, target
    ),
    weight="length"
)

astar_time = time.perf_counter() - start
astar_dist = path_length_m(G, astar_path)

print(f"A* time: {astar_time:.6f} s")
print(f"A* distance: {astar_dist:.0f} m")

m_astar = base_map(center, kfupm, ithra)

m_astar = add_route_to_map(
    m_astar,
    G,
    astar_path,
    "A*",
    color="purple"
)

display(m_astar)

A* time: 0.002655 s
A* distance: 6243 m


Simulated Annealing
Max. Iterations: 250, Starting Temp.: 1, Geometric Cooling with alpha = 0.995

In [11]:
def random_dfs_path(G, source, target):
    stack = [(source, [source])]
    visited = set()

    while stack:
        node, path = stack.pop()

        if node == target:
            return path

        if node in visited:
            continue

        visited.add(node)

        neighbors = list(G.successors(node))
        random.shuffle(neighbors)

        for nbr in neighbors:
            if nbr not in visited:
                stack.append((nbr, path + [nbr]))

    raise nx.NetworkXNoPath("No path found")


def simulated_annealing_route(G, source, target,
                              iters=400,
                              T0=10.0,
                              alpha=0.995,
                              seed=7):

    rnd = random.Random(seed)

    current = random_dfs_path(G, source, target)
    current_cost = path_length_m(G, current)

    best = current
    best_cost = current_cost

    T = T0

    for _ in range(iters):

        if len(current) < 6:
            break

        i = rnd.randint(1, len(current) - 4)
        j = rnd.randint(i + 2, len(current) - 2)

        a = current[i]
        b = current[j]

        try:
            mid = nx.shortest_path(G, source=a, target=b, weight="length")
        except nx.NetworkXNoPath:
            T *= alpha
            continue

        candidate = current[:i] + mid + current[j+1:]
        cand_cost = path_length_m(G, candidate)

        delta = cand_cost - current_cost

        if delta < 0 or rnd.random() < math.exp(-delta / T):
            current = candidate
            current_cost = cand_cost

            if cand_cost < best_cost:
                best = candidate
                best_cost = cand_cost

        T *= alpha

    return best

start = time.perf_counter()
sa_path = simulated_annealing_route(G, src, dst,
                                    iters=400,
                                    T0=10.0,
                                    alpha=0.995,
                                    seed=7)

sa_time = time.perf_counter() - start
sa_dist = path_length_m(G, sa_path)

print(f"SA time: {sa_time:.6f} s")
print(f"SA distance: {sa_dist:.0f} m")

m_sa = base_map(center, kfupm, ithra)
m_sa = add_route_to_map(m_sa, G, sa_path, "Simulated Annealing", color="purple")
m_sa

SA time: 0.078826 s
SA distance: 6543 m


Final comparison (Time & Distance)

In [15]:
results = [
    ("BFS", bfs_time, bfs_dist, len(bfs_path)),
    ("DFS", dfs_time, dfs_dist, len(dfs_path)),
    ("Dijkstra", dijkstra_time, dijkstra_dist, len(dijkstra_path)),
    (
        "Hill Climbing with Backtracking",
        hill_time,
        hill_dist,
        len(hill_path)
    ),
    ("A*", astar_time, astar_dist, len(astar_path)),
    ("Simulated Annealing", sa_time, sa_dist, len(sa_path)),
]

print(
    f"{'Algorithm':<34} "
    f"{'Time(s)':>10} "
    f"{'Distance(m)':>14} "
    f"{'Nodes in Path':>14}"
)

print("-" * 76)

for name, t, d, n_nodes in results:
    print(
        f"{name:<34} "
        f"{t:>10.6f} "
        f"{d:>14.0f} "
        f"{n_nodes:>14}"
    )

Algorithm                             Time(s)    Distance(m)  Nodes in Path
----------------------------------------------------------------------------
BFS                                  0.000458           6492             26
DFS                                  0.001104          37514            268
Dijkstra                             0.001704           6243             36
Hill Climbing with Backtracking      0.000387          12346            108
A*                                   0.002655           6243             36
Simulated Annealing                  0.078826           6543             39


#Conclusion
Hill Climbing with Backtracking was the fastest algorithm (0.000387 s), but it produced a much longer route of 12,346 m. This shows that faster execution does not necessarily provide a high-quality route.

BFS was also very fast (0.000458 s) and produced a 6,492 m route, which was 249 m longer than the optimal route because BFS does not consider edge distances.

DFS performed poorly, producing the longest route (37,514 m) despite its low execution time.

Dijkstra and A* both produced the shortest route of 6,243 m using 36 nodes. However, Dijkstra was faster (0.001704 s) than A* (0.002655 s) for this relatively small graph.

Simulated Annealing produced a near-optimal route of 6,543 m, only 300 m longer than the optimal route, but required significantly more computation time (0.078826 s).

Overall, Dijkstra provided the best balance between route quality and execution time, while A* achieved the same optimal distance but did not provide a speed advantage for this small network.